<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_8_%E2%80%94_NDVI_DYNAMICS_AND_VEGETATION_CONDITION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
```python
# =============================================================================
# SECTION 4.8 — NDVI DYNAMICS AND VEGETATION CONDITION
# TEZPUR–NORTH LAKHIMPUR NATIONAL HIGHWAY CORRIDOR
# =============================================================================
#
# Purpose:
#   Analyse Sentinel-2 NDVI between 2016 and 2025 and generate manuscript-ready
#   statistics for:
#
#   Figure 13:
#       TZPR_NLP_NDVI_2016
#       TZPR_NLP_NDVI_2020
#       TZPR_NLP_NDVI_2025
#
#   Figure 14:
#       TZPR_NLP_NDVI_Change_2016_2025
#
# Main analyses:
#   1. NDVI descriptive statistics for 2016, 2020 and 2025
#   2. NDVI condition classes
#   3. NDVI 2016–2025 change statistics
#   4. Area of NDVI increase/decrease/stable conditions
#   5. Magnitude of NDVI decline/increase
#   6. Strong NDVI decline (< -0.15)
#   7. Strong NDVI increase (> +0.15)
#   8. Annualized NDVI change
#   9. Raster quality-control information
#  10. Manuscript-ready tables and text
#
# INPUT DIRECTORY:
#   /content/drive/MyDrive/TZPR_NLP_Research
#
# OUTPUT DIRECTORY:
#   /content/drive/MyDrive/TZPR_NLP_Research/NDVI_Analysis_4_8
#
# =============================================================================

import os
import math
import warnings
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# =============================================================================
# 1. USER SETTINGS
# =============================================================================

INPUT_DIR = "/content/drive/MyDrive/TZPR_NLP_Research"

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "NDVI_Analysis_4_8"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# Input files
# -------------------------------------------------------------------------

NDVI_FILES = {
    2016: os.path.join(
        INPUT_DIR,
        "TZPR_NLP_NDVI_2016.tif"
    ),

    2020: os.path.join(
        INPUT_DIR,
        "TZPR_NLP_NDVI_2020.tif"
    ),

    2025: os.path.join(
        INPUT_DIR,
        "TZPR_NLP_NDVI_2025.tif"
    ),

    "CHANGE_2016_2025": os.path.join(
        INPUT_DIR,
        "TZPR_NLP_NDVI_Change_2016_2025.tif"
    )
}

# -------------------------------------------------------------------------
# Analysis settings
# -------------------------------------------------------------------------

# Strong NDVI decline threshold used in your development-pressure workflow
STRONG_DECLINE_THRESHOLD = -0.15

# Strong NDVI increase threshold
STRONG_INCREASE_THRESHOLD = 0.15

# Stable threshold:
# values between -0.05 and +0.05 are treated as relatively stable
STABLE_THRESHOLD = 0.05

# Chunk size for large rasters
CHUNK_ROWS = 256

# Expected NDVI physical range
NDVI_MIN_VALID = -1.0
NDVI_MAX_VALID = 1.0


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def check_input_files():
    """
    Verify all required GeoTIFFs are present.
    """

    print("\n" + "=" * 80)
    print("CHECKING INPUT FILES")
    print("=" * 80)

    missing = []

    for label, path in NDVI_FILES.items():

        if os.path.exists(path):

            size_mb = os.path.getsize(path) / (1024 ** 2)

            print(
                f"[OK] {label}: "
                f"{os.path.basename(path)} "
                f"({size_mb:.2f} MB)"
            )

        else:

            print(
                f"[MISSING] {label}: "
                f"{os.path.basename(path)}"
            )

            missing.append(path)

    if missing:

        print("\nERROR: Required input files are missing:")

        for path in missing:
            print("  -", path)

        raise FileNotFoundError(
            "One or more required NDVI GeoTIFF files are missing."
        )


def geographic_pixel_area_ha(transform, height):
    """
    Calculate approximate area of each raster row in hectares for EPSG:4326.

    Formula:
        A = R^2 * dlon * [sin(lat2) - sin(lat1)]

    Returns:
        area array of length = raster height
    """

    R = 6371008.8

    dlon = abs(transform.a)

    dlat = abs(transform.e)

    lat_centers = np.array([
        transform.f + transform.e * (row + 0.5)
        for row in range(height)
    ])

    lat1 = np.radians(lat_centers - dlat / 2.0)
    lat2 = np.radians(lat_centers + dlat / 2.0)

    area_m2 = (
        R ** 2
        * np.radians(dlon)
        * (
            np.sin(lat2)
            -
            np.sin(lat1)
        )
    )

    area_ha = area_m2 / 10000.0

    return area_ha


def read_raster_metadata(path):

    with rasterio.open(path) as src:

        metadata = {
            "file": os.path.basename(path),
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "dtype": src.dtypes[0],
            "crs": str(src.crs),
            "transform": str(src.transform),
            "nodata": src.nodata,
            "resolution_x": abs(src.transform.a),
            "resolution_y": abs(src.transform.e),
            "bounds_left": src.bounds.left,
            "bounds_right": src.bounds.right,
            "bounds_bottom": src.bounds.bottom,
            "bounds_top": src.bounds.top
        }

    return metadata


def validate_raster_alignment(paths):

    print("\n" + "=" * 80)
    print("CHECKING RASTER ALIGNMENT")
    print("=" * 80)

    reference = None

    for label, path in paths.items():

        with rasterio.open(path) as src:

            current = {
                "width": src.width,
                "height": src.height,
                "crs": src.crs,
                "transform": src.transform
            }

            if reference is None:

                reference = current

                print(
                    f"[REFERENCE] {label}: "
                    f"{src.width} × {src.height}"
                )

            else:

                problems = []

                if current["width"] != reference["width"]:
                    problems.append("width")

                if current["height"] != reference["height"]:
                    problems.append("height")

                if current["crs"] != reference["crs"]:
                    problems.append("CRS")

                if current["transform"] != reference["transform"]:
                    problems.append("transform")

                if problems:

                    raise ValueError(
                        f"Raster alignment problem in {label}: "
                        + ", ".join(problems)
                    )

                print(
                    f"[OK] {label}: "
                    f"{src.width} × {src.height}"
                )

    print("All rasters are spatially aligned.")


def update_stats(stats, values, area_values):

    """
    Update descriptive statistics using valid pixel values.
    """

    if values.size == 0:
        return

    stats["valid_pixels"] += values.size

    stats["sum"] += np.sum(values, dtype=np.float64)

    stats["sum_sq"] += np.sum(
        values.astype(np.float64) ** 2
    )

    stats["min"] = min(
        stats["min"],
        float(np.min(values))
    )

    stats["max"] = max(
        stats["max"],
        float(np.max(values))
    )

    stats["valid_area_ha"] += np.sum(
        area_values,
        dtype=np.float64
    )


def calculate_ndvi_statistics(path):

    print(
        f"\nAnalysing: {os.path.basename(path)}"
    )

    with rasterio.open(path) as src:

        area_by_row = geographic_pixel_area_ha(
            src.transform,
            src.height
        )

        stats = {
            "valid_pixels": 0,
            "sum": 0.0,
            "sum_sq": 0.0,
            "min": np.inf,
            "max": -np.inf,
            "valid_area_ha": 0.0
        }

        # For exact median and percentile statistics,
        # collect valid values.
        all_values = []

        for row_start in range(
            0,
            src.height,
            CHUNK_ROWS
        ):

            row_end = min(
                row_start + CHUNK_ROWS,
                src.height
            )

            arr = src.read(
                1,
                window=rasterio.windows.Window(
                    0,
                    row_start,
                    src.width,
                    row_end - row_start
                )
            ).astype(np.float32)

            valid = (
                np.isfinite(arr)
                &
                (arr >= NDVI_MIN_VALID)
                &
                (arr <= NDVI_MAX_VALID)
            )

            if src.nodata is not None:

                valid &= (
                    arr != src.nodata
                )

            values = arr[valid]

            if values.size == 0:
                continue

            row_area = np.repeat(
                area_by_row[row_start:row_end],
                src.width
            )

            valid_area = row_area[valid.ravel()]

            update_stats(
                stats,
                values,
                valid_area
            )

            all_values.append(
                values.astype(np.float32)
            )

        if stats["valid_pixels"] == 0:

            raise ValueError(
                f"No valid NDVI pixels found in {path}"
            )

        values = np.concatenate(
            all_values
        ).astype(np.float64)

        mean = (
            stats["sum"]
            /
            stats["valid_pixels"]
        )

        variance = (
            stats["sum_sq"]
            /
            stats["valid_pixels"]
            -
            mean ** 2
        )

        variance = max(
            variance,
            0
        )

        std = math.sqrt(
            variance
        )

        result = {
            "valid_pixels": stats["valid_pixels"],
            "area_ha": stats["valid_area_ha"],
            "area_km2": stats["valid_area_ha"] / 100.0,
            "minimum": stats["min"],
            "maximum": stats["max"],
            "mean": mean,
            "median": float(np.median(values)),
            "std": std,
            "p05": float(np.percentile(values, 5)),
            "p25": float(np.percentile(values, 25)),
            "p75": float(np.percentile(values, 75)),
            "p95": float(np.percentile(values, 95))
        }

        return result


# =============================================================================
# 3. NDVI CONDITION CLASSIFICATION
# =============================================================================

def ndvi_condition_statistics(path):

    print(
        f"\nCalculating NDVI condition classes:"
        f" {os.path.basename(path)}"
    )

    class_definitions = {

        "Very low (<0.0)": (
            lambda x: x < 0.0
        ),

        "Low (0.0–0.20)": (
            lambda x: (x >= 0.0) & (x < 0.20)
        ),

        "Moderate (0.20–0.40)": (
            lambda x: (x >= 0.20) & (x < 0.40)
        ),

        "High (0.40–0.60)": (
            lambda x: (x >= 0.40) & (x < 0.60)
        ),

        "Very high (≥0.60)": (
            lambda x: x >= 0.60
        )
    }

    counts = {
        key: 0
        for key in class_definitions
    }

    areas = {
        key: 0.0
        for key in class_definitions
    }

    with rasterio.open(path) as src:

        area_by_row = geographic_pixel_area_ha(
            src.transform,
            src.height
        )

        for row_start in range(
            0,
            src.height,
            CHUNK_ROWS
        ):

            row_end = min(
                row_start + CHUNK_ROWS,
                src.height
            )

            arr = src.read(
                1,
                window=rasterio.windows.Window(
                    0,
                    row_start,
                    src.width,
                    row_end - row_start
                )
            ).astype(np.float32)

            valid = (
                np.isfinite(arr)
                &
                (arr >= NDVI_MIN_VALID)
                &
                (arr <= NDVI_MAX_VALID)
            )

            if src.nodata is not None:
                valid &= (
                    arr != src.nodata
                )

            if not np.any(valid):
                continue

            row_area = np.repeat(
                area_by_row[row_start:row_end],
                src.width
            )

            flat = arr.ravel()
            valid_flat = valid.ravel()

            values = flat[valid_flat]
            pixel_area = row_area[valid_flat]

            for class_name, condition in class_definitions.items():

                mask = condition(values)

                counts[class_name] += int(
                    np.sum(mask)
                )

                areas[class_name] += float(
                    np.sum(pixel_area[mask])
                )

    total_area = sum(areas.values())

    rows = []

    for class_name in class_definitions:

        area_ha = areas[class_name]

        rows.append({

            "NDVI_Class": class_name,

            "Pixels": counts[class_name],

            "Area_ha": area_ha,

            "Area_km2": area_ha / 100.0,

            "Percent_of_valid_area":
                (
                    area_ha /
                    total_area *
                    100.0
                )
                if total_area > 0
                else np.nan
        })

    return pd.DataFrame(rows)


# =============================================================================
# 4. NDVI CHANGE STATISTICS
# =============================================================================

def calculate_change_statistics(path):

    print(
        f"\nAnalysing NDVI change:"
        f" {os.path.basename(path)}"
    )

    stats = {

        "valid_pixels": 0,

        "sum": 0.0,

        "sum_sq": 0.0,

        "min": np.inf,

        "max": -np.inf,

        "valid_area_ha": 0.0,

        "increase_pixels": 0,

        "decrease_pixels": 0,

        "stable_pixels": 0,

        "increase_area_ha": 0.0,

        "decrease_area_ha": 0.0,

        "stable_area_ha": 0.0,

        "strong_decline_pixels": 0,

        "strong_increase_pixels": 0,

        "strong_decline_area_ha": 0.0,

        "strong_increase_area_ha": 0.0,

        "moderate_decline_area_ha": 0.0,

        "moderate_increase_area_ha": 0.0
    }

    all_values = []

    with rasterio.open(path) as src:

        area_by_row = geographic_pixel_area_ha(
            src.transform,
            src.height
        )

        for row_start in range(
            0,
            src.height,
            CHUNK_ROWS
        ):

            row_end = min(
                row_start + CHUNK_ROWS,
                src.height
            )

            arr = src.read(
                1,
                window=rasterio.windows.Window(
                    0,
                    row_start,
                    src.width,
                    row_end - row_start
                )
            ).astype(np.float32)

            valid = np.isfinite(arr)

            if src.nodata is not None:

                valid &= (
                    arr != src.nodata
                )

            # Reasonable NDVI change range
            valid &= (
                arr >= -2.0
            ) & (
                arr <= 2.0
            )

            if not np.any(valid):
                continue

            row_area = np.repeat(
                area_by_row[row_start:row_end],
                src.width
            )

            values = arr.ravel()[valid.ravel()]

            pixel_area = row_area[
                valid.ravel()
            ]

            stats["valid_pixels"] += values.size

            stats["sum"] += np.sum(
                values,
                dtype=np.float64
            )

            stats["sum_sq"] += np.sum(
                values.astype(np.float64) ** 2
            )

            stats["min"] = min(
                stats["min"],
                float(np.min(values))
            )

            stats["max"] = max(
                stats["max"],
                float(np.max(values))
            )

            stats["valid_area_ha"] += np.sum(
                pixel_area
            )

            all_values.append(
                values
            )

            # -------------------------------------------------------------
            # Increase / decrease / stable
            # -------------------------------------------------------------

            decrease = (
                values < -STABLE_THRESHOLD
            )

            increase = (
                values > STABLE_THRESHOLD
            )

            stable = ~(
                decrease | increase
            )

            stats["decrease_pixels"] += int(
                np.sum(decrease)
            )

            stats["increase_pixels"] += int(
                np.sum(increase)
            )

            stats["stable_pixels"] += int(
                np.sum(stable)
            )

            stats["decrease_area_ha"] += np.sum(
                pixel_area[decrease]
            )

            stats["increase_area_ha"] += np.sum(
                pixel_area[increase]
            )

            stats["stable_area_ha"] += np.sum(
                pixel_area[stable]
            )

            # -------------------------------------------------------------
            # Strong decline / strong increase
            # -------------------------------------------------------------

            strong_decline = (
                values <
                STRONG_DECLINE_THRESHOLD
            )

            strong_increase = (
                values >
                STRONG_INCREASE_THRESHOLD
            )

            stats["strong_decline_pixels"] += int(
                np.sum(strong_decline)
            )

            stats["strong_increase_pixels"] += int(
                np.sum(strong_increase)
            )

            stats["strong_decline_area_ha"] += np.sum(
                pixel_area[strong_decline]
            )

            stats["strong_increase_area_ha"] += np.sum(
                pixel_area[strong_increase]
            )

            # Moderate change
            moderate_decline = (
                (values <= -STABLE_THRESHOLD)
                &
                (values >= STRONG_DECLINE_THRESHOLD)
            )

            moderate_increase = (
                (values >= STABLE_THRESHOLD)
                &
                (values <= STRONG_INCREASE_THRESHOLD)
            )

            stats["moderate_decline_area_ha"] += np.sum(
                pixel_area[moderate_decline]
            )

            stats["moderate_increase_area_ha"] += np.sum(
                pixel_area[moderate_increase]
            )

    values = np.concatenate(
        all_values
    ).astype(np.float64)

    mean = (
        stats["sum"]
        /
        stats["valid_pixels"]
    )

    variance = (
        stats["sum_sq"]
        /
        stats["valid_pixels"]
        -
        mean ** 2
    )

    std = math.sqrt(
        max(variance, 0)
    )

    total_area = stats["valid_area_ha"]

    result = {

        "valid_pixels":
            stats["valid_pixels"],

        "area_ha":
            total_area,

        "area_km2":
            total_area / 100.0,

        "minimum":
            stats["min"],

        "maximum":
            stats["max"],

        "mean":
            mean,

        "median":
            float(np.median(values)),

        "std":
            std,

        "increase_area_ha":
            stats["increase_area_ha"],

        "decrease_area_ha":
            stats["decrease_area_ha"],

        "stable_area_ha":
            stats["stable_area_ha"],

        "increase_percent":
            stats["increase_area_ha"]
            /
            total_area
            *
            100.0,

        "decrease_percent":
            stats["decrease_area_ha"]
            /
            total_area
            *
            100.0,

        "stable_percent":
            stats["stable_area_ha"]
            /
            total_area
            *
            100.0,

        "strong_decline_area_ha":
            stats["strong_decline_area_ha"],

        "strong_increase_area_ha":
            stats["strong_increase_area_ha"],

        "strong_decline_percent":
            stats["strong_decline_area_ha"]
            /
            total_area
            *
            100.0,

        "strong_increase_percent":
            stats["strong_increase_area_ha"]
            /
            total_area
            *
            100.0,

        "moderate_decline_area_ha":
            stats["moderate_decline_area_ha"],

        "moderate_increase_area_ha":
            stats["moderate_increase_area_ha"]
    }

    return result


# =============================================================================
# 5. HISTOGRAM GENERATION
# =============================================================================

def create_ndvi_histogram(path, year):

    print(
        f"Creating NDVI histogram: {year}"
    )

    with rasterio.open(path) as src:

        arr = src.read(1).astype(np.float32)

        valid = (
            np.isfinite(arr)
            &
            (arr >= -1)
            &
            (arr <= 1)
        )

        if src.nodata is not None:

            valid &= (
                arr != src.nodata
            )

        values = arr[valid]

    plt.figure(
        figsize=(9, 6)
    )

    plt.hist(
        values,
        bins=50,
        edgecolor="black"
    )

    plt.xlabel("NDVI")

    plt.ylabel("Pixel Frequency")

    plt.title(
        f"NDVI Distribution — {year}"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    output = os.path.join(
        OUTPUT_DIR,
        f"NDVI_Histogram_{year}.png"
    )

    plt.savefig(
        output,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


def create_change_histogram(path):

    print(
        "Creating NDVI change histogram"
    )

    with rasterio.open(path) as src:

        arr = src.read(1).astype(np.float32)

        valid = (
            np.isfinite(arr)
            &
            (arr >= -2)
            &
            (arr <= 2)
        )

        if src.nodata is not None:

            valid &= (
                arr != src.nodata
            )

        values = arr[valid]

    plt.figure(
        figsize=(9, 6)
    )

    plt.hist(
        values,
        bins=60,
        edgecolor="black"
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1.5,
        label="No change"
    )

    plt.axvline(
        STRONG_DECLINE_THRESHOLD,
        linestyle=":",
        linewidth=1.5,
        label="Strong decline threshold"
    )

    plt.axvline(
        STRONG_INCREASE_THRESHOLD,
        linestyle=":",
        linewidth=1.5,
        label="Strong increase threshold"
    )

    plt.xlabel("NDVI Change (2016–2025)")

    plt.ylabel("Pixel Frequency")

    plt.title(
        "Distribution of NDVI Change, 2016–2025"
    )

    plt.legend()

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    output = os.path.join(
        OUTPUT_DIR,
        "NDVI_Change_Histogram_2016_2025.png"
    )

    plt.savefig(
        output,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# =============================================================================
# 6. CREATE MANUSCRIPT TABLES
# =============================================================================

def create_manuscript_outputs(
    yearly_stats,
    change_stats,
    condition_tables
):

    print("\n" + "=" * 80)
    print("CREATING MANUSCRIPT TABLES")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Table 1 — Descriptive NDVI statistics
    # -------------------------------------------------------------------------

    rows = []

    for year in [2016, 2020, 2025]:

        s = yearly_stats[year]

        rows.append({

            "Year": year,

            "Mean_NDVI":
                s["mean"],

            "Median_NDVI":
                s["median"],

            "Minimum_NDVI":
                s["minimum"],

            "Maximum_NDVI":
                s["maximum"],

            "Standard_Deviation":
                s["std"],

            "P05":
                s["p05"],

            "P25":
                s["p25"],

            "P75":
                s["p75"],

            "P95":
                s["p95"],

            "Area_ha":
                s["area_ha"],

            "Area_km2":
                s["area_km2"],

            "Valid_Pixels":
                s["valid_pixels"]
        })

    df_yearly = pd.DataFrame(rows)

    df_yearly.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "01_NDVI_Descriptive_Statistics_2016_2025.csv"
        ),
        index=False
    )

    # -------------------------------------------------------------------------
    # Table 2 — NDVI condition classes
    # -------------------------------------------------------------------------

    condition_all = []

    for year, table in condition_tables.items():

        temp = table.copy()

        temp.insert(
            0,
            "Year",
            year
        )

        condition_all.append(
            temp
        )

    df_condition = pd.concat(
        condition_all,
        ignore_index=True
    )

    df_condition.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "02_NDVI_Condition_Classes_2016_2020_2025.csv"
        ),
        index=False
    )

    # -------------------------------------------------------------------------
    # Table 3 — NDVI change
    # -------------------------------------------------------------------------

    total_area = change_stats["area_ha"]

    df_change = pd.DataFrame([{

        "Period":
            "2016–2025",

        "Mean_NDVI_Change":
            change_stats["mean"],

        "Median_NDVI_Change":
            change_stats["median"],

        "Minimum_Change":
            change_stats["minimum"],

        "Maximum_Change":
            change_stats["maximum"],

        "Std_Change":
            change_stats["std"],

        "Increase_Area_ha":
            change_stats["increase_area_ha"],

        "Increase_Area_km2":
            change_stats["increase_area_ha"] / 100,

        "Increase_Percent":
            change_stats["increase_percent"],

        "Decrease_Area_ha":
            change_stats["decrease_area_ha"],

        "Decrease_Area_km2":
            change_stats["decrease_area_ha"] / 100,

        "Decrease_Percent":
            change_stats["decrease_percent"],

        "Stable_Area_ha":
            change_stats["stable_area_ha"],

        "Stable_Area_km2":
            change_stats["stable_area_ha"] / 100,

        "Stable_Percent":
            change_stats["stable_percent"],

        "Strong_Decline_Area_ha":
            change_stats["strong_decline_area_ha"],

        "Strong_Decline_Percent":
            change_stats["strong_decline_percent"],

        "Strong_Increase_Area_ha":
            change_stats["strong_increase_area_ha"],

        "Strong_Increase_Percent":
            change_stats["strong_increase_percent"],

        "Total_Valid_Area_ha":
            total_area
    }])

    df_change.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "03_NDVI_Change_Statistics_2016_2025.csv"
        ),
        index=False
    )

    # -------------------------------------------------------------------------
    # Table 4 — Annualized change
    # -------------------------------------------------------------------------

    years = 2025 - 2016

    mean_change = change_stats["mean"]

    df_annual = pd.DataFrame([{

        "Period":
            "2016–2025",

        "Years":
            years,

        "Mean_NDVI_Change":
            mean_change,

        "Annualized_Mean_NDVI_Change":
            mean_change / years,

        "Gross_Decline_Area_ha":
            change_stats["decrease_area_ha"],

        "Gross_Increase_Area_ha":
            change_stats["increase_area_ha"],

        "Stable_Area_ha":
            change_stats["stable_area_ha"]
    }])

    df_annual.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "04_Annualized_NDVI_Change.csv"
        ),
        index=False
    )

    # -------------------------------------------------------------------------
    # Table 5 — Compact manuscript table
    # -------------------------------------------------------------------------

    compact = []

    for year in [2016, 2020, 2025]:

        s = yearly_stats[year]

        compact.append({

            "Indicator":
                f"Mean NDVI ({year})",

            "Value":
                s["mean"],

            "Unit":
                "NDVI"
        })

    compact.extend([

        {
            "Indicator":
                "Mean NDVI change (2016–2025)",

            "Value":
                change_stats["mean"],

            "Unit":
                "NDVI"
        },

        {
            "Indicator":
                "Median NDVI change",

            "Value":
                change_stats["median"],

            "Unit":
                "NDVI"
        },

        {
            "Indicator":
                "NDVI decrease area",

            "Value":
                change_stats["decrease_area_ha"],

            "Unit":
                "ha"
        },

        {
            "Indicator":
                "NDVI decrease",

            "Value":
                change_stats["decrease_percent"],

            "Unit":
                "%"
        },

        {
            "Indicator":
                "NDVI increase area",

            "Value":
                change_stats["increase_area_ha"],

            "Unit":
                "ha"
        },

        {
            "Indicator":
                "NDVI increase",

            "Value":
                change_stats["increase_percent"],

            "Unit":
                "%"
        },

        {
            "Indicator":
                "Stable NDVI area",

            "Value":
                change_stats["stable_area_ha"],

            "Unit":
                "ha"
        },

        {
            "Indicator":
                "Stable NDVI",

            "Value":
                change_stats["stable_percent"],

            "Unit":
                "%"
        },

        {
            "Indicator":
                "Strong NDVI decline",

            "Value":
                change_stats["strong_decline_area_ha"],

            "Unit":
                "ha"
        },

        {
            "Indicator":
                "Strong NDVI decline",

            "Value":
                change_stats["strong_decline_percent"],

            "Unit":
                "%"
        },

        {
            "Indicator":
                "Strong NDVI increase",

            "Value":
                change_stats["strong_increase_area_ha"],

            "Unit":
                "ha"
        },

        {
            "Indicator":
                "Strong NDVI increase",

            "Value":
                change_stats["strong_increase_percent"],

            "Unit":
                "%"
        },

        {
            "Indicator":
                "Annualized mean NDVI change",

            "Value":
                change_stats["mean"] / 9,

            "Unit":
                "NDVI yr⁻¹"
        }
    ])

    df_compact = pd.DataFrame(
        compact
    )

    df_compact.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "05_NDVI_Manuscript_Summary_Table.csv"
        ),
        index=False
    )

    return (
        df_yearly,
        df_condition,
        df_change,
        df_annual,
        df_compact
    )


# =============================================================================
# 7. MANUSCRIPT TEXT VALUES
# =============================================================================

def create_manuscript_values(
    yearly_stats,
    change_stats
):

    s16 = yearly_stats[2016]
    s20 = yearly_stats[2020]
    s25 = yearly_stats[2025]

    mean16 = s16["mean"]
    mean20 = s20["mean"]
    mean25 = s25["mean"]

    mean_change = change_stats["mean"]

    relative_change_percent = (
        mean_change /
        abs(mean16)
        *
        100.0
    )

    annual_change = (
        mean_change /
        9.0
    )

    text = f"""
===============================================================================
SECTION 4.8 — MANUSCRIPT VALUES
NDVI DYNAMICS AND VEGETATION CONDITION
===============================================================================

FIGURE 13
Spatial distribution of Sentinel-2 NDVI across the highway corridor.

NDVI 2016
---------
Mean NDVI       : {mean16:.4f}
Median NDVI     : {s16["median"]:.4f}
Minimum NDVI    : {s16["minimum"]:.4f}
Maximum NDVI    : {s16["maximum"]:.4f}
Std. deviation  : {s16["std"]:.4f}
Area            : {s16["area_ha"]:.2f} ha
Area            : {s16["area_km2"]:.2f} km²


NDVI 2020
---------
Mean NDVI       : {mean20:.4f}
Median NDVI     : {s20["median"]:.4f}
Minimum NDVI    : {s20["minimum"]:.4f}
Maximum NDVI    : {s20["maximum"]:.4f}
Std. deviation  : {s20["std"]:.4f}
Area            : {s20["area_ha"]:.2f} ha
Area            : {s20["area_km2"]:.2f} km²


NDVI 2025
---------
Mean NDVI       : {mean25:.4f}
Median NDVI     : {s25["median"]:.4f}
Minimum NDVI    : {s25["minimum"]:.4f}
Maximum NDVI    : {s25["maximum"]:.4f}
Std. deviation  : {s25["std"]:.4f}
Area            : {s25["area_ha"]:.2f} ha
Area            : {s25["area_km2"]:.2f} km²


FIGURE 14
Spatial distribution of NDVI change between 2016 and 2025.

Mean NDVI change
----------------
Mean change             : {mean_change:.4f}
Median change           : {change_stats["median"]:.4f}
Minimum change          : {change_stats["minimum"]:.4f}
Maximum change          : {change_stats["maximum"]:.4f}
Standard deviation      : {change_stats["std"]:.4f}

NDVI decrease
-------------
Area                    : {change_stats["decrease_area_ha"]:.2f} ha
Area                    : {change_stats["decrease_area_ha"]/100:.2f} km²
Percentage              : {change_stats["decrease_percent"]:.2f} %

NDVI increase
-------------
Area                    : {change_stats["increase_area_ha"]:.2f} ha
Area                    : {change_stats["increase_area_ha"]/100:.2f} km²
Percentage              : {change_stats["increase_percent"]:.2f} %

Relatively stable NDVI
----------------------
Area                    : {change_stats["stable_area_ha"]:.2f} ha
Area                    : {change_stats["stable_area_ha"]/100:.2f} km²
Percentage              : {change_stats["stable_percent"]:.2f} %

Strong NDVI decline
-------------------
Threshold               : NDVI change < {STRONG_DECLINE_THRESHOLD}
Area                    : {change_stats["strong_decline_area_ha"]:.2f} ha
Area                    : {change_stats["strong_decline_area_ha"]/100:.2f} km²
Percentage              : {change_stats["strong_decline_percent"]:.2f} %

Strong NDVI increase
--------------------
Threshold               : NDVI change > +{STRONG_INCREASE_THRESHOLD}
Area                    : {change_stats["strong_increase_area_ha"]:.2f} ha
Area                    : {change_stats["strong_increase_area_ha"]/100:.2f} km²
Percentage              : {change_stats["strong_increase_percent"]:.2f} %

Temporal change
---------------
2016 mean NDVI          : {mean16:.4f}
2020 mean NDVI          : {mean20:.4f}
2025 mean NDVI          : {mean25:.4f}

2016–2025 mean change   : {mean_change:.4f}
Annualized change       : {annual_change:.5f} NDVI yr⁻¹

Relative change from 2016:
{relative_change_percent:.2f} %

===============================================================================
INTERPRETATION REMINDER
===============================================================================

1. Negative NDVI change indicates a reduction in vegetation greenness between
   the two endpoint years.

2. Positive NDVI change indicates an increase in vegetation greenness.

3. NDVI decline should not automatically be interpreted as permanent
   vegetation loss because NDVI responds to seasonality, cropping cycles,
   moisture, phenology and land-cover condition.

4. The NDVI change raster represents endpoint change between 2016 and 2025;
   it is not an annual trend analysis.

5. Avoid claiming direct highway causality from NDVI change alone.

6. Strong decline is defined here as NDVI change < -0.15, consistent with the
   development-pressure analysis.

===============================================================================
"""

    output = os.path.join(
        OUTPUT_DIR,
        "06_NDVI_Manuscript_Values.txt"
    )

    with open(
        output,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(text)

    return text


# =============================================================================
# 8. RASTER INFORMATION TABLE
# =============================================================================

def create_raster_information():

    rows = []

    for label, path in NDVI_FILES.items():

        metadata = read_raster_metadata(
            path
        )

        metadata["Dataset"] = label

        rows.append(
            metadata
        )

    df = pd.DataFrame(
        rows
    )

    df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "07_NDVI_Raster_Information.csv"
        ),
        index=False
    )

    return df


# =============================================================================
# 9. MAIN PROCESS
# =============================================================================

def main():

    print("\n")
    print("=" * 80)
    print("SECTION 4.8 — NDVI DYNAMICS AND VEGETATION CONDITION")
    print("TEZPUR–NORTH LAKHIMPUR HIGHWAY CORRIDOR")
    print("=" * 80)

    print(
        "\nInput directory:"
    )

    print(
        INPUT_DIR
    )

    print(
        "\nOutput directory:"
    )

    print(
        OUTPUT_DIR
    )

    # -------------------------------------------------------------------------
    # Check inputs
    # -------------------------------------------------------------------------

    check_input_files()

    # -------------------------------------------------------------------------
    # Check alignment
    # -------------------------------------------------------------------------

    validate_raster_alignment(
        NDVI_FILES
    )

    # -------------------------------------------------------------------------
    # Raster metadata
    # -------------------------------------------------------------------------

    create_raster_information()

    # -------------------------------------------------------------------------
    # Yearly NDVI statistics
    # -------------------------------------------------------------------------

    yearly_stats = {}

    for year in [2016, 2020, 2025]:

        yearly_stats[year] = (
            calculate_ndvi_statistics(
                NDVI_FILES[year]
            )
        )

    # -------------------------------------------------------------------------
    # NDVI condition statistics
    # -------------------------------------------------------------------------

    condition_tables = {}

    for year in [2016, 2020, 2025]:

        condition_tables[year] = (
            ndvi_condition_statistics(
                NDVI_FILES[year]
            )
        )

    # -------------------------------------------------------------------------
    # NDVI change
    # -------------------------------------------------------------------------

    change_stats = (
        calculate_change_statistics(
            NDVI_FILES["CHANGE_2016_2025"]
        )
    )

    # -------------------------------------------------------------------------
    # Create tables
    # -------------------------------------------------------------------------

    (
        df_yearly,
        df_condition,
        df_change,
        df_annual,
        df_compact
    ) = create_manuscript_outputs(
        yearly_stats,
        change_stats,
        condition_tables
    )

    # -------------------------------------------------------------------------
    # Create manuscript values
    # -------------------------------------------------------------------------

    manuscript_text = create_manuscript_values(
        yearly_stats,
        change_stats
    )

    # -------------------------------------------------------------------------
    # Create histograms
    # -------------------------------------------------------------------------

    create_ndvi_histogram(
        NDVI_FILES[2016],
        2016
    )

    create_ndvi_histogram(
        NDVI_FILES[2020],
        2020
    )

    create_ndvi_histogram(
        NDVI_FILES[2025],
        2025
    )

    create_change_histogram(
        NDVI_FILES["CHANGE_2016_2025"]
    )

    # -------------------------------------------------------------------------
    # Console summary
    # -------------------------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("NDVI ANALYSIS SUMMARY")
    print("=" * 80)

    print(
        "\nMean NDVI:"
    )

    for year in [2016, 2020, 2025]:

        print(
            f"  {year}: "
            f"{yearly_stats[year]['mean']:.4f}"
        )

    print(
        "\nMean NDVI change 2016–2025:"
    )

    print(
        f"  {change_stats['mean']:.4f}"
    )

    print(
        "\nMedian NDVI change:"
    )

    print(
        f"  {change_stats['median']:.4f}"
    )

    print(
        "\nNDVI decrease:"
    )

    print(
        f"  {change_stats['decrease_area_ha']:.2f} ha "
        f"({change_stats['decrease_percent']:.2f}%)"
    )

    print(
        "\nNDVI increase:"
    )

    print(
        f"  {change_stats['increase_area_ha']:.2f} ha "
        f"({change_stats['increase_percent']:.2f}%)"
    )

    print(
        "\nRelatively stable:"
    )

    print(
        f"  {change_stats['stable_area_ha']:.2f} ha "
        f"({change_stats['stable_percent']:.2f}%)"
    )

    print(
        "\nStrong NDVI decline (< -0.15):"
    )

    print(
        f"  {change_stats['strong_decline_area_ha']:.2f} ha "
        f"({change_stats['strong_decline_percent']:.2f}%)"
    )

    print(
        "\nStrong NDVI increase (> +0.15):"
    )

    print(
        f"  {change_stats['strong_increase_area_ha']:.2f} ha "
        f"({change_stats['strong_increase_percent']:.2f}%)"
    )

    print("\n")
    print("=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)

    for filename in sorted(
        os.listdir(OUTPUT_DIR)
    ):

        print(
            "  ",
            filename
        )

    print("\nNDVI Section 4.8 analysis completed successfully.")


# =============================================================================
# 10. RUN
# =============================================================================

if __name__ == "__main__":

    main()
```
